In [2]:
import os
import shutil
import numpy as np
import scipy.signal as signal
from scipy.interpolate import interp1d
import wfdb

# =========================================================================
# Global Signal & Segmentation Constants
# =========================================================================
TARGET_FS = 128          # Harmonized sampling rate (128 Hz)
BEAT_LEN = 128           # 128-dimensional raw beat vector
LEFT_SAMPLES = 38        # ~300 ms prior to R-peak
RIGHT_SAMPLES = 90       # ~700 ms after R-peak
HRV_WIN_SEC = 60.0       # Rolling HRV window duration
HRV_STEP_SEC = 30.0      # Rolling HRV window step stride

BASE_STORAGE_DIR = "./ecg_raw_128dim_dataset"

DATASET_SPLITS = {
    'train': {
        # Standard DS1 partition: mix of baseline (100s) and complex ectopic rhythms (200s)
        'mitdb': [
            '101', '106', '108', '109', '112', '114', '115', '116', '118', '119', '122', '124',
            '201', '203', '205', '207', '208', '209', '215', '220', '223', '230'
        ],
        # 7 subjects (~60% of healthy cohort)
        'nsrdb': [
            '16265', '16272', '16273', '16420', '16483', '16539', '16773'
        ],
        # 7 subjects (~47% of CHF cohort)
        'chfdb': [
            'chf01', 'chf02', 'chf03', 'chf04', 'chf05', 'chf06', 'chf07'
        ]
    },
    'test': {
        # Standard DS2 partition: completely unseen subjects for rigorous evaluation
        'mitdb': [
            '100', '103', '105', '111', '113', '117', '121', '123',
            '200', '202', '210', '212', '213', '214', '219', '221', '222', '228', '231', '232', '233', '234'
        ],
        # 5 unseen healthy subjects
        'nsrdb': [
            '16786', '16795', '17052', '17453', '18177'
        ],
        # 8 unseen CHF subjects
        'chfdb': [
            'chf08', 'chf09', 'chf10', 'chf11', 'chf12', 'chf13', 'chf14', 'chf15'
        ]
    }
}

NORMAL_SYMBOLS = {'N', 'L', 'R', 'e', 'j'}
ARRHYTHMIA_SYMBOLS = {'A', 'a', 'J', 'S', 'V', 'E', 'F'}
REJECT_SYMBOLS = {'|', '~', '+', 'Q', 'x'}

# =========================================================================
# Signal Preprocessing Engine (Polyphase Filtering + Standardization)
# =========================================================================
def resample_and_filter(raw_sig, orig_fs, target_fs=TARGET_FS):
    if orig_fs == 360 and target_fs == 128:
        sig_resampled = signal.resample_poly(raw_sig, up=32, down=90)
    elif orig_fs == 250 and target_fs == 128:
        sig_resampled = signal.resample_poly(raw_sig, up=64, down=125)
    elif orig_fs == target_fs:
        sig_resampled = raw_sig.copy()
    else:
        sig_resampled = signal.resample(raw_sig, int(len(raw_sig) * target_fs / orig_fs))

    nyq = 0.5 * target_fs
    b, a = signal.butter(3, [0.5 / nyq, 40.0 / nyq], btype='band')
    filtered = signal.filtfilt(b, a, sig_resampled)
    return (filtered - np.mean(filtered)) / (np.std(filtered) + 1e-8)

# =========================================================================
# Continuous Non-Constant HRV Timeline Engine
# =========================================================================
def compute_continuous_hrv_timeline(r_peaks, symbols, total_duration_sec, win_len=HRV_WIN_SEC, step=HRV_STEP_SEC):
    peak_times = r_peaks / TARGET_FS
    rr_all = np.diff(peak_times)

    # Filter physiological extremes (< 250 ms or > 2500 ms)
    valid_rr_mask = (rr_all >= 0.25) & (rr_all <= 2.5)

    is_sinus = np.array([s in NORMAL_SYMBOLS for s in symbols])
    nn_valid_mask = is_sinus[:-1] & is_sinus[1:] & valid_rr_mask

    timeline = []
    n_steps = int(max(1, (total_duration_sec - win_len) // step + 1))
    patient_median_rr = float(np.median(rr_all[valid_rr_mask])) if np.any(valid_rr_mask) else 0.85

    for s in range(n_steps):
        t_start = s * step
        t_end = t_start + win_len

        idx_in_win = np.where((peak_times[1:] >= t_start) & (peak_times[1:] < t_end))[0]
        if len(idx_in_win) == 0:
            timeline.append([patient_median_rr, 0.04, 0.03, 2.0, 150.0, 150.0, 1.0, 0.0])
            continue

        win_symbols = symbols[idx_in_win]
        n_total = len(win_symbols)
        n_normal = sum(1 for sym in win_symbols if sym in NORMAL_SYMBOLS)
        ectopic_burden = float(1.0 - (n_normal / (n_total + 1e-6)))

        # Prioritize pure NN intervals; fall back to all valid RR in heavy arrhythmic runs
        sinus_in_win = idx_in_win[nn_valid_mask[idx_in_win]]
        if len(sinus_in_win) >= 5:
            intervals = rr_all[sinus_in_win]
        else:
            actual_valid = idx_in_win[valid_rr_mask[idx_in_win]]
            intervals = rr_all[actual_valid] if len(actual_valid) >= 2 else np.array([patient_median_rr])

        mean_nn = float(np.mean(intervals))
        sdnn = float(np.std(intervals)) if len(intervals) > 1 else 0.01

        diffs = np.diff(intervals)
        if len(diffs) > 0:
            rmssd = float(np.sqrt(np.mean(diffs ** 2)))
            pnn50 = float(np.sum(np.abs(diffs) > 0.05) / len(diffs) * 100.0)
        else:
            rmssd = 0.01
            pnn50 = 0.0

        # Spectral domain calculations (Welch PSD on 4 Hz resampled series)
        if len(intervals) >= 4 and (np.sum(intervals) > 4.0):
            t_raw = np.cumsum(intervals)
            f_interp = interp1d(t_raw, intervals, kind='linear', fill_value='extrapolate')
            t_uniform = np.arange(t_raw[0], t_raw[-1], 0.25)

            if len(t_uniform) >= 16:
                nn_uniform = f_interp(t_uniform)
                freqs, psd = signal.welch(nn_uniform - np.mean(nn_uniform), fs=4.0, nperseg=min(len(nn_uniform), 64))

                lf_mask = (freqs >= 0.04) & (freqs < 0.15)
                hf_mask = (freqs >= 0.15) & (freqs < 0.40)

                # Updated for NumPy 2.0+:
                lf_power = float(np.trapezoid(psd[lf_mask], freqs[lf_mask])) if np.any(lf_mask) else 1.0
                hf_power = float(np.trapezoid(psd[hf_mask], freqs[hf_mask])) if np.any(hf_mask) else 1.0
                lf_hf = float(lf_power / (hf_power + 1e-6))
            else:
                lf_power, hf_power, lf_hf = 1.0, 1.0, 1.0
        else:
            approx_power = float(max(1.0, (sdnn * 1000.0) ** 2 / 2.0))
            lf_power, hf_power, lf_hf = approx_power, approx_power, 1.0

        timeline.append([mean_nn, sdnn, rmssd, pnn50, lf_power, hf_power, lf_hf, ectopic_burden])

    return np.array(timeline, dtype=np.float32)

# =========================================================================
# Record Processor
# =========================================================================
def extract_record_data(record_path, orig_fs, ann_ext, max_beats=None):
    rec = wfdb.rdrecord(record_path)
    ann = wfdb.rdann(record_path, ann_ext)

    norm_sig = resample_and_filter(rec.p_signal[:, 0], orig_fs=orig_fs, target_fs=TARGET_FS)
    scale = TARGET_FS / float(orig_fs)
    r_peaks = np.round(ann.sample * scale).astype(int)
    symbols = np.array(ann.symbol)

    if max_beats is not None:
        r_peaks = r_peaks[:max_beats]
        symbols = symbols[:max_beats]

    total_dur_sec = len(norm_sig) / TARGET_FS
    hrv_timeline = compute_continuous_hrv_timeline(r_peaks, symbols, total_dur_sec)

    extracted = {'c0': [], 'c1': [], 'c2': []}
    rr_all = np.diff(r_peaks) / TARGET_FS

    for i in range(1, len(r_peaks) - 1):
        sym = symbols[i]
        if sym in REJECT_SYMBOLS:
            continue

        start = r_peaks[i] - LEFT_SAMPLES
        end = r_peaks[i] + RIGHT_SAMPLES
        if start < 0 or end > len(norm_sig) or (end - start) != BEAT_LEN:
            continue

        waveform = norm_sig[start:end]
        pre_rr = (r_peaks[i] - r_peaks[i - 1]) / TARGET_FS
        post_rr = (r_peaks[i + 1] - r_peaks[i]) / TARGET_FS
        rr_ratio = pre_rr / (post_rr + 1e-6)

        s_idx = max(0, (i - 1) - 10)
        local_mean = np.mean(rr_all[s_idx:(i - 1)]) if (i - 1) > s_idx else pre_rr
        rel_rr = pre_rr / (local_mean + 1e-6)
        rr_feats = [pre_rr, post_rr, rr_ratio, rel_rr]

        # Look up continuous HRV window for this beat's timestamp
        beat_t = r_peaks[i] / TARGET_FS
        w_idx = int(np.clip(beat_t // HRV_STEP_SEC, 0, len(hrv_timeline) - 1))
        hrv_feats = hrv_timeline[w_idx]

        # Route by source database and annotation symbol
        if 'chf' in os.path.basename(record_path):
            target_cls = 'c1' if sym in ARRHYTHMIA_SYMBOLS else 'c2'
        elif 'mitdb' in os.path.dirname(record_path):
            target_cls = 'c1' if sym in ARRHYTHMIA_SYMBOLS else 'c0'
        else:  # nsrdb
            target_cls = 'c0'

        extracted[target_cls].append((waveform, rr_feats, hrv_feats))

    return extracted

# =========================================================================
# Storage & Orchestration Engine
# =========================================================================
def init_storage(base_dir):
    if os.path.exists(base_dir):
        shutil.rmtree(base_dir)
    for split in ['train', 'test']:
        for cls_name in ['class_0_normal', 'class_1_arrhythmia', 'class_2_chf']:
            os.makedirs(os.path.join(base_dir, split, cls_name), exist_ok=True)

def append_or_save(file_path, new_data):
    if len(new_data) == 0:
        return
    new_arr = np.array(new_data, dtype=np.float32)
    if os.path.exists(file_path):
        existing = np.load(file_path)
        combined = np.concatenate([existing, new_arr], axis=0)
        np.save(file_path, combined)
    else:
        np.save(file_path, new_arr)

def write_extracted_split(split, cls_key, folder_name, triplets):
    if not triplets:
        return
    raw_beats = [t[0] for t in triplets]
    rr_feats = [t[1] for t in triplets]
    hrv_feats = [t[2] for t in triplets]

    dest_dir = os.path.join(BASE_STORAGE_DIR, split, folder_name)
    append_or_save(os.path.join(dest_dir, 'beats_raw.npy'), raw_beats)
    append_or_save(os.path.join(dest_dir, 'beats_rr.npy'), rr_feats)
    append_or_save(os.path.join(dest_dir, 'beats_hrv.npy'), hrv_feats)

def run_pipeline(mitdb_dir, nsrdb_dir, chfdb_dir):
    init_storage(BASE_STORAGE_DIR)

    for split in ['train', 'test']:
        print(f"\n{'='*55}\n PROCESSING DATASET SPLIT: {split.upper()}\n{'='*55}")

        # 1. MIT-BIH Arrhythmia Database (360 Hz)
        print(">> Processing MITDB...")
        for rec_id in DATASET_SPLITS[split]['mitdb']:
            rec_path = os.path.join(mitdb_dir, rec_id)
            if not os.path.exists(rec_path + '.hea'):
                continue
            res = extract_record_data(rec_path, orig_fs=360, ann_ext='atr')
            write_extracted_split(split, 'c0', 'class_0_normal', res['c0'])
            write_extracted_split(split, 'c1', 'class_1_arrhythmia', res['c1'])

        # 2. Normal Sinus Rhythm Database (128 Hz)
        print(">> Processing NSRDB...")
        for rec_id in DATASET_SPLITS[split]['nsrdb']:
            rec_path = os.path.join(nsrdb_dir, rec_id)
            if not os.path.exists(rec_path + '.hea'):
                continue
            res = extract_record_data(rec_path, orig_fs=128, ann_ext='atr', max_beats=4000)
            write_extracted_split(split, 'c0', 'class_0_normal', res['c0'])

        # 3. Congestive Heart Failure Database (250 Hz)
        print(">> Processing CHFDB...")
        for rec_id in DATASET_SPLITS[split]['chfdb']:
            rec_path = os.path.join(chfdb_dir, rec_id)
            if not os.path.exists(rec_path + '.hea'):
                continue
            res = extract_record_data(rec_path, orig_fs=250, ann_ext='ecg', max_beats=4000)
            write_extracted_split(split, 'c2', 'class_2_chf', res['c2'])
            write_extracted_split(split, 'c1', 'class_1_arrhythmia', res['c1'])

    # Final Inventory Verification
    print(f"\n{'='*70}\nDATASET STORAGE AUDIT & DIMENSION VERIFICATION\n{'='*70}")
    for split in ['train', 'test']:
        print(f"[{split.upper()}]")
        for cls_name in ['class_0_normal', 'class_1_arrhythmia', 'class_2_chf']:
            p = os.path.join(BASE_STORAGE_DIR, split, cls_name)
            rf = os.path.join(p, 'beats_raw.npy')
            rrf = os.path.join(p, 'beats_rr.npy')
            hf = os.path.join(p, 'beats_hrv.npy')

            n_raw = len(np.load(rf)) if os.path.exists(rf) else 0
            n_rr = len(np.load(rrf)) if os.path.exists(rrf) else 0
            n_hrv = len(np.load(hf)) if os.path.exists(hf) else 0

            raw_shape = np.load(rf).shape if n_raw > 0 else (0, 0)
            rr_shape = np.load(rrf).shape if n_rr > 0 else (0, 0)
            hrv_shape = np.load(hf).shape if n_hrv > 0 else (0, 0)

            print(f"  {cls_name:<20}: Beats={raw_shape} | RR={rr_shape} | HRV={hrv_shape}")
    print("=" * 70 + "\n")

if __name__ == '__main__':
    MITDB_PATH = "./mitdb"
    NSRDB_PATH = "./nsrdb"
    CHFDB_PATH = "./chfdb"

    run_pipeline(MITDB_PATH, NSRDB_PATH, CHFDB_PATH)


 PROCESSING DATASET SPLIT: TRAIN
>> Processing MITDB...
>> Processing NSRDB...
>> Processing CHFDB...

 PROCESSING DATASET SPLIT: TEST
>> Processing MITDB...
>> Processing NSRDB...
>> Processing CHFDB...

DATASET STORAGE AUDIT & DIMENSION VERIFICATION
[TRAIN]
  class_0_normal      : Beats=(72194, 128) | RR=(72194, 4) | HRV=(72194, 8)
  class_1_arrhythmia  : Beats=(5502, 128) | RR=(5502, 4) | HRV=(5502, 8)
  class_2_chf         : Beats=(27593, 128) | RR=(27593, 4) | HRV=(27593, 8)
[TEST]
  class_0_normal      : Beats=(64324, 128) | RR=(64324, 4) | HRV=(64324, 8)
  class_1_arrhythmia  : Beats=(5506, 128) | RR=(5506, 4) | HRV=(5506, 8)
  class_2_chf         : Beats=(31906, 128) | RR=(31906, 4) | HRV=(31906, 8)

